# 01 — Exploración del dataset y definición de clases

Análisis inicial del dataset antes de cualquier preprocesamiento.

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed/train')
EXTENSIONS    = {'.jpg', '.jpeg', '.png'}

## Celda 2 — Distribución de clases originales

In [ ]:
raw_counts = {}
for folder in sorted(RAW_DIR.iterdir()):
    if folder.is_dir():
        count = sum(1 for f in folder.iterdir() if f.suffix.lower() in EXTENSIONS)
        raw_counts[folder.name] = count

df_raw = pd.DataFrame(raw_counts.items(), columns=['clase_original', 'n_imagenes'])
df_raw = df_raw.sort_values('n_imagenes', ascending=False).reset_index(drop=True)

print(df_raw.to_string(index=False))
print(f"\nTotal: {df_raw['n_imagenes'].sum()} imágenes en {len(df_raw)} clases")

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(df_raw['clase_original'], df_raw['n_imagenes'], color='steelblue', edgecolor='white')
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_title('Distribución de clases originales (data/raw)', fontsize=13)
ax.set_xlabel('Clase')
ax.set_ylabel('N° imágenes')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../results/figures/01_distribucion_clases_originales.png', dpi=150)
plt.show()

## Celda 3 — Distribución de clases fusionadas

In [ ]:
fused_counts = {}
for folder in sorted(PROCESSED_DIR.iterdir()):
    if folder.is_dir():
        count = sum(1 for f in folder.iterdir() if f.suffix.lower() in EXTENSIONS)
        fused_counts[folder.name] = count

df_fused = pd.DataFrame(fused_counts.items(), columns=['clase_final', 'n_imagenes'])
df_fused = df_fused.sort_values('n_imagenes', ascending=False).reset_index(drop=True)

print(df_fused.to_string(index=False))
print(f"\nTotal train: {df_fused['n_imagenes'].sum()} imágenes en {len(df_fused)} clases")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars0 = axes[0].bar(df_raw['clase_original'], df_raw['n_imagenes'], color='steelblue', edgecolor='white')
axes[0].bar_label(bars0, padding=3, fontsize=8)
axes[0].set_title('Clases originales (12)', fontsize=12)
axes[0].set_xlabel('Clase')
axes[0].set_ylabel('N° imágenes')
axes[0].tick_params(axis='x', rotation=35)

bars1 = axes[1].bar(df_fused['clase_final'], df_fused['n_imagenes'], color='seagreen', edgecolor='white')
axes[1].bar_label(bars1, padding=3, fontsize=8)
axes[1].set_title('Clases fusionadas (9)', fontsize=12)
axes[1].set_xlabel('Clase')
axes[1].set_ylabel('N° imágenes')
axes[1].tick_params(axis='x', rotation=35)

plt.suptitle('Comparación: clases originales vs fusionadas (split train)', fontsize=13)
plt.tight_layout()
plt.savefig('../results/figures/01_comparacion_clases_original_vs_fusionado.png', dpi=150)
plt.show()

desbalance = df_fused['n_imagenes'].max() / df_fused['n_imagenes'].min()
print(f"\nRatio de desbalance (max/min): {desbalance:.2f}x")

## Celda 4 — Ejemplos visuales por clase

In [ ]:
clases    = sorted([f.name for f in PROCESSED_DIR.iterdir() if f.is_dir()])
N_EJEMPLOS = 5

fig, axes = plt.subplots(len(clases), N_EJEMPLOS, figsize=(N_EJEMPLOS * 3, len(clases) * 3))

for i, clase in enumerate(clases):
    imagenes = sorted((PROCESSED_DIR / clase).iterdir())[:N_EJEMPLOS]
    for j, img_path in enumerate(imagenes):
        img = Image.open(img_path).convert('RGB')
        axes[i, j].imshow(img)
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_ylabel(clase, fontsize=11, fontweight='bold', rotation=0, labelpad=60, va='center')

plt.suptitle('Ejemplos visuales por clase (5 imágenes cada una)', fontsize=14)
plt.tight_layout()
plt.savefig('../results/figures/01_ejemplos_visuales_por_clase.png', dpi=150)
plt.show()

## Celda 5 — Análisis de resoluciones

In [ ]:
widths  = []
heights = []

for clase in clases:
    carpeta  = PROCESSED_DIR / clase
    imagenes = list(carpeta.iterdir())[:50]
    for img_path in imagenes:
        if img_path.suffix.lower() in EXTENSIONS:
            with Image.open(img_path) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)

print('Estadísticas de resolución:')
print(f'  Ancho  — min: {min(widths)}  max: {max(widths)}  media: {np.mean(widths):.0f}')
print(f'  Alto   — min: {min(heights)}  max: {max(heights)}  media: {np.mean(heights):.0f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(widths, bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(256, color='red', linestyle='--', label='target 256px')
axes[0].set_title('Distribución de anchos')
axes[0].set_xlabel('Píxeles')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

axes[1].hist(heights, bins=30, color='seagreen', edgecolor='white')
axes[1].axvline(256, color='red', linestyle='--', label='target 256px')
axes[1].set_title('Distribución de altos')
axes[1].set_xlabel('Píxeles')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.suptitle('Distribución de resoluciones en el dataset', fontsize=13)
plt.tight_layout()
plt.savefig('../results/figures/01_distribucion_resoluciones.png', dpi=150)
plt.show()

## Celda 6 — Histogramas de color por clase (RGB)

In [ ]:
N_MUESTRAS = 80
canales    = ['R', 'G', 'B']
colores    = ['red', 'green', 'blue']

fig, axes = plt.subplots(len(clases), 3, figsize=(12, len(clases) * 2.5))

for i, clase in enumerate(clases):
    carpeta   = PROCESSED_DIR / clase
    imagenes  = sorted(carpeta.iterdir())[:N_MUESTRAS]
    hist_acum = [np.zeros(256), np.zeros(256), np.zeros(256)]

    for img_path in imagenes:
        if img_path.suffix.lower() in EXTENSIONS:
            img = np.array(Image.open(img_path).convert('RGB'))
            for c in range(3):
                hist, _ = np.histogram(img[:, :, c], bins=256, range=(0, 256))
                hist_acum[c] += hist

    for c in range(3):
        hist_acum[c] /= hist_acum[c].sum()
        axes[i, c].plot(hist_acum[c], color=colores[c], linewidth=1.2)
        axes[i, c].set_xlim(0, 255)
        axes[i, c].set_ylim(0)
        axes[i, c].set_title(f'{clase} — canal {canales[c]}', fontsize=9)
        axes[i, c].tick_params(labelsize=7)
        if c == 0:
            axes[i, c].set_ylabel('Densidad', fontsize=8)

plt.suptitle('Histogramas de color promedio por clase (RGB)', fontsize=13)
plt.tight_layout()
plt.savefig('../results/figures/01_histogramas_color_por_clase.png', dpi=150)
plt.show()

## Celda 7 — Observaciones y conclusiones

### Distribución de clases

- El dataset original tiene 12 clases con fuerte desbalance: `clothes` domina con 5,325 imágenes mientras `brown-glass` tiene solo 607.
- Tras la fusión a 9 clases, `textile` absorbió `clothes` y `shoes` acumulando 5,111 imágenes, con un ratio de desbalance de ~10x respecto a `trash` (488).
- Se aplicó balanceo a 750 imágenes por clase en train mediante submuestreo (textile, glass) y data augmentation (resto). Val y test se mantienen con datos reales sin modificar.
- Las transformaciones de augmentation aplicadas fueron: flip horizontal, rotación ±15°, ajuste de brillo ±20%, zoom leve y ruido gaussiano leve.

### Análisis de histogramas de color RGB

- **Battery y glass** presentan picos fuertes en valores altos (200-255) en los tres canales, indicando que el fondo blanco domina el histograma. Ambas clases serán difíciles de separar solo por color.
- **Cardboard y paper** muestran distribuciones similares en tonos medios-altos. Alta probabilidad de confusión entre sí.
- **Plastic** tiene distribución amplia similar a cardboard. Se espera confusión entre estas tres clases por color.
- **Metal** presenta densidad muy baja y distribución plana en todo el rango, producto de la reflectancia metálica. El color no es discriminativo para esta clase.
- **Organic** es la clase más oscura, concentrada en tonos bajos-medios. Buena separabilidad respecto al resto.
- **Textile** tiene la distribución más plana y ancha, cubriendo todo el rango. El color es completamente inútil como descriptor para esta clase.
- **Trash** muestra pico en valores altos en R y B pero casi ausente en G.

### Conclusión crítica

El color RGB por sí solo no es suficiente para separar las 9 clases. Los grupos problemáticos son:

- **Glass / battery / paper / plastic:** histogramas muy similares, dominados por fondos claros.
- **Cardboard / paper / plastic:** tonos neutros medios-altos casi indistinguibles por color.
- **Textile:** imposible clasificar por color dado su variabilidad intrínseca.

Esto confirma la necesidad de descriptores de textura (LBP, Gabor) y forma, y justifica el pipeline híbrido con verificador CNN.

### Decisiones tomadas

- 9 clases finales confirmadas: battery, cardboard, glass, metal, organic, paper, plastic, textile, trash.
- Resize target: 256×256 px.
- Balanceo: 750 imágenes por clase en train.
- Val y test sin augmentation para preservar evaluación sobre datos reales.
- Los descriptores de textura y forma son imprescindibles dado el bajo poder discriminativo del color para varias clases.